# 합격자소서 임베딩 파이프라인 — 전체 재구축
**BGE-M3 → ChromaDB (chromadb 1.x 신규 생성)**

### 업로드할 파일
- `essays.db` 1개만 올리면 됩니다 (chroma_db.zip 불필요)

### 실행 순서
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. Cell 1 ~ 7 순서대로 실행
3. Cell 7 완료 후 `chroma_db.zip` + `essays.db` 자동 다운로드

In [ ]:
# ── Cell 1: 패키지 설치 ─────────────────────────────────────
!pip install -q FlagEmbedding chromadb

In [ ]:
# ── Cell 2: GPU + 버전 확인 ──────────────────────────────────
import torch, chromadb
print('GPU   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 — T4로 변경하세요')
print('chromadb:', chromadb.__version__)

In [ ]:
# ── Cell 3: essays.db 업로드 ─────────────────────────────────
from google.colab import files
import os

print('essays.db 선택 → 업로드')
uploaded = files.upload()
assert 'essays.db' in uploaded, 'essays.db 를 선택하세요'
print(f'essays.db: {os.path.getsize("essays.db"):,} bytes')

In [ ]:
# ── Cell 4: 모델 로드 ───────────────────────────────────────
from FlagEmbedding import BGEM3FlagModel

MODEL_NAME = 'BAAI/bge-m3'
print('모델 로드 중...')
model = BGEM3FlagModel(MODEL_NAME, use_fp16=True)
print('모델 로드 완료')

In [ ]:
# ── Cell 5: ChromaDB 새로 생성 ───────────────────────────────
import chromadb, shutil, os

CHROMA_PATH = '/content/chroma_db'
COLLECTION  = 'cover_letters'

# 혹시 남아있는 이전 데이터 제거
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print('기존 chroma_db 삭제')

client     = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(
    name=COLLECTION,
    metadata={'hnsw:space': 'cosine'},
)
print(f'ChromaDB 새로 생성 완료 | 현재 벡터: {collection.count()}개')

In [ ]:
# ── Cell 6: 전체 임베딩 실행 ─────────────────────────────────
import sqlite3, time
from datetime import datetime
from tqdm.auto import tqdm

conn = sqlite3.connect('essays.db')

COLS = ['qna_id','essay_id','question_clean','answer_clean','question_type',
        'char_count','source','company','org_type','role','hire_type',
        'year','season','university','major']

rows = conn.execute("""
    SELECT
        q.id, q.essay_id,
        q.question_clean, q.answer_clean,
        q.question_type, q.char_count,
        e.source, e.company, e.org_type,
        e.role, e.hire_type, e.year, e.season,
        e.university, e.major
    FROM  qna q
    JOIN  essays e ON e.id = q.essay_id
    WHERE q.is_valid = 1
      AND q.question_clean IS NOT NULL
    ORDER BY q.id
""").fetchall()

pending = [dict(zip(COLS, r)) for r in rows]
print(f'임베딩 대상: {len(pending):,}건')

# ── 임베딩 + ChromaDB 저장 ──────────────────────────────────
BATCH_SIZE = 64
done = failed = 0
t0   = time.time()

def make_text(q, a):
    q = (q or '').strip()
    a = (a or '').strip()
    return f'{q}\n{a}' if q else a

def mark_done(conn, qna_ids, chroma_ids):
    now = datetime.now().isoformat()
    conn.executemany(
        'INSERT OR REPLACE INTO qna_embeddings (qna_id, chroma_id, model_name, embedded_at) VALUES (?,?,?,?)',
        [(qid, cid, MODEL_NAME, now) for qid, cid in zip(qna_ids, chroma_ids)],
    )
    conn.commit()

for start in tqdm(range(0, len(pending), BATCH_SIZE), desc='임베딩'):
    batch = pending[start : start + BATCH_SIZE]
    texts = [make_text(r['question_clean'], r['answer_clean']) for r in batch]

    try:
        out  = model.encode(texts, batch_size=BATCH_SIZE, max_length=8192)
        vecs = out['dense_vecs']
    except Exception as e:
        print(f'임베딩 오류 (idx={start}): {e}')
        failed += len(batch)
        continue

    chroma_ids = [f"qna_{r['qna_id']}" for r in batch]
    qna_ids    = [r['qna_id'] for r in batch]
    metadatas  = [
        {
            'qna_id':        int(r['qna_id']),
            'essay_id':      int(r['essay_id']),
            'source':        r['source']        or '',
            'company':       r['company']        or '',
            'org_type':      r['org_type']       or '',
            'role':          r['role']           or '',
            'hire_type':     r['hire_type']      or '',
            'year':          r['year']           or '',
            'season':        r['season']         or '',
            'question_type': r['question_type']  or '',
            'university':    r['university']     or '',
            'major':         r['major']          or '',
            'char_count':    int(r['char_count'] or 0),
        }
        for r in batch
    ]

    try:
        collection.upsert(
            ids=chroma_ids,
            embeddings=vecs.tolist(),
            metadatas=metadatas,
            documents=texts,
        )
        mark_done(conn, qna_ids, chroma_ids)
        done += len(batch)
    except Exception as e:
        print(f'upsert 오류 (idx={start}): {e}')
        failed += len(batch)

elapsed = time.time() - t0
print(f'\n완료: 성공={done:,}건  실패={failed}건  소요={elapsed/60:.1f}분')
print(f'ChromaDB 총 벡터: {collection.count():,}개')

In [ ]:
# ── Cell 7: 다운로드 ─────────────────────────────────────────
import shutil, os
from google.colab import files

conn.close()

# chroma_db 압축
shutil.make_archive('/content/chroma_db', 'zip', '/content/chroma_db')
print(f'chroma_db.zip: {os.path.getsize("/content/chroma_db.zip")/1024/1024:.1f} MB')

# 다운로드
files.download('/content/chroma_db.zip')  # → 로컬 프로젝트 루트에 압축 해제
files.download('/content/essays.db')      # → 로컬 essays.db 교체

## 다운로드 후 로컬에서 할 일

```powershell
# 1. 기존 chroma_db 삭제
Remove-Item -Recurse -Force chroma_db

# 2. 새로 받은 zip 압축 해제 (프로젝트 루트에서 실행)
Expand-Archive chroma_db.zip -DestinationPath chroma_db

# 3. essays.db 교체 (다운받은 파일로)
# → 탐색기에서 essays.db를 프로젝트 루트로 복사

# 4. 데모 실행
python -X utf8 demo.py --company 삼성전자
```